In [ ]:
%%capture
# Updated: January 2026 - Latest package versions
# Note: If you've already run install.sh, these packages are already installed
!pip install llama-index==0.14.13 cohere==5.20.2 openai==2.15.0 llama-index-embeddings-openai==0.6.13 llama-index-llms-cohere llama-index-vector-stores-qdrant==0.9.1 qdrant-client==1.16.2 

In [1]:
# Standard library imports
import os
from getpass import getpass
import nest_asyncio

# Third-party imports
from dotenv import load_dotenv

# Apply nest_asyncio to allow nested event loops (needed for Jupyter notebooks)
# This is required when using async operations in Jupyter
nest_asyncio.apply()

# Load environment variables from .env file
# This will read API keys and other variables from the .env file in the project root
load_dotenv()

True

In [2]:
# Get Cohere API key from environment variable
# Falls back to prompting user if not found in .env file
# Note: Using os.getenv() is safer than os.environ[] as it returns None instead of raising KeyError
CO_API_KEY = os.getenv("CO_API_KEY") or getpass("Enter your Cohere API key: ")

In [3]:
# Get OpenAI API key from environment variable
# Falls back to prompting user if not found in .env file
# Note: Using os.getenv() is safer than os.environ[] as it returns None instead of raising KeyError
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")

In [4]:
# Get Qdrant URL from environment variable
# Qdrant can run locally or in the cloud (Qdrant Cloud)
# Example local: "http://localhost:6333"
# Example cloud: "https://your-cluster.qdrant.io"
QDRANT_URL = os.getenv("QDRANT_URL") or getpass("Enter your Qdrant URL: ")

In [5]:
# Get Qdrant API key from environment variable
# Required for Qdrant Cloud, optional for local Qdrant instances
# Falls back to prompting user if not found in .env file
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY") or getpass("Enter your Qdrant API Key: ")

In [6]:
# Import Path from pathlib for cross-platform path handling
from pathlib import Path

def create_directory(directory_name):
    """
    Create a directory if it doesn't exist.
    
    Parameters:
    - directory_name: Name/path of the directory to create
    """
    path = Path(directory_name)
    # parents=True creates parent directories if needed
    # exist_ok=True doesn't raise an error if directory already exists
    path.mkdir(parents=True, exist_ok=True)
    print(f"Directory '{directory_name}' created successfully.")

# Create the data directory for storing downloaded files
create_directory("dataM2_02_05")

Directory 'dataM2_02_05' created successfully.


In [7]:
# Download a text file from Project Gutenberg
# Note: wget works on Unix/Linux/Mac. For Windows, use PowerShell or Python requests
# Alternative Python approach (cross-platform):
import requests
from pathlib import Path

url = "https://www.gutenberg.org/cache/epub/10763/pg10763.txt"
file_path = Path("dataM2_02_05/pg10763.txt")

try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    file_path.write_text(response.text, encoding='utf-8')
    print(f"Downloaded {file_path.name} to {file_path}")
except requests.RequestException as e:
    print(f"Failed to download: {e}")

# Unix/Linux/Mac alternative (uncomment if needed):
# !wget -P data https://www.gutenberg.org/cache/epub/10763/pg10763.txt 

Downloaded pg10763.txt to dataM2_02_05\pg10763.txt


# 🗄️ Storing

Loading and indexing data costs time and money.

By default, indexed data is stored in memory. But, you can store your data to avoid the time and costs associated with re-indexing them.  The simplest way to do this **persisting to disk**.

Each `Index` object has a `.persist()` method, which will write all the data to disk at the specified location.

Now that we've dowloaded data, let's:

1) Load as Document
2) Parse as Nodes
3) Create index

In [8]:
# Import SimpleDirectoryReader for loading documents from files
from llama_index.core import SimpleDirectoryReader

# Specify the file path to load
file_path = "dataM2_02_05/pg10763.txt"

# Load document from a specific file
# input_files: List of specific files to load (instead of loading entire directory)
# filename_as_id: Use the filename as the document ID (useful for tracking sources)
document = SimpleDirectoryReader(
    input_files=[file_path], 
    filename_as_id=True
).load_data()

In [9]:
# Import SentenceSplitter for custom document chunking
from llama_index.core.node_parser import SentenceSplitter

# Create a node parser with custom settings
# chunk_size: Maximum tokens per chunk (512 tokens ≈ 400 words)
# chunk_overlap: Overlapping tokens between chunks (maintains context)
# paragraph_separator: Prefer splitting at quadruple newlines (section breaks)
sentence_splitter = SentenceSplitter(
    chunk_size=512,  # Larger chunks for better context retention
    chunk_overlap=16,  # Overlap to maintain context between chunks
    paragraph_separator="\n\n\n\n"  # Split at section breaks when possible
)

In [10]:
# Instantiate embedding model
# Using OpenAI embeddings as per course objective
# Note: If you encounter 429 (rate limit) errors, wait a few minutes and retry
# or consider using Cohere embeddings as an alternative
from llama_index.embeddings.openai import OpenAIEmbedding

# Use OpenAI embedding model
# text-embedding-3-small: Latest model, 1536 dimensions, cost-effective
# Always pass api_key explicitly or it will read from environment
embed_model = OpenAIEmbedding(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small"  # Latest OpenAI embedding model
)

# ☁️ Using a Vector Database

We'll use qdrant as our vector database of choice throughout this course.

To use qdrant to store embeddings from the `VectorStoreIndex`, you need to:

- Initialize the qdrant client

- Create a `Collection` to store your data in qdrant

- Assign qdrant as the `vector_store` in a `StorageContext`

- Initialize your `VectorStoreIndex` using that `StorageContext`

Below, we initialize a `QdrantClient` for interacting with qdrant, an open-source vector store. 


In [11]:
# Import Qdrant client and vector store integration
import qdrant_client
from llama_index.vector_stores.qdrant import QdrantVectorStore

# Initialize Qdrant client
# QdrantClient connects to your Qdrant instance (local or cloud)
# url: Qdrant server URL (local: "http://localhost:6333" or cloud URL)
# api_key: Required for Qdrant Cloud, optional for local instances
client = qdrant_client.QdrantClient(
    url=QDRANT_URL, 
    api_key=QDRANT_API_KEY if QDRANT_API_KEY else None,  # Only pass if provided
)

# Create QdrantVectorStore
# This connects LlamaIndex to Qdrant for storing and retrieving embeddings
# collection_name: Name of the Qdrant collection (creates if doesn't exist)
# embed_model: Embedding model used (needed to determine vector dimensions)
vector_store = QdrantVectorStore(
    client=client, 
    collection_name="it_can_be_done",  # Collection name in Qdrant
    embed_model=embed_model,  # Embedding model for dimension matching
)

2026-01-28 17:59:53,208 - INFO - HTTP Request: GET https://63bfc1a0-2d12-4262-8028-7f8cb5e33277.europe-west3-0.gcp.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
2026-01-28 17:59:53,667 - INFO - HTTP Request: GET https://63bfc1a0-2d12-4262-8028-7f8cb5e33277.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/it_can_be_done/exists "HTTP/1.1 200 OK"


# 🗃️ Storage Context

`StorageContext` in `LlamaIndex` is a core abstraction that revolves around the storage of `Nodes`, indices, and vectors.  It facilitates data storage and retrieval.

It is a utility container that supports the following:

 - `docstore`: A [`BaseDocumentStore`](https://github.com/run-llama/llama_index/blob/main/llama-index-core/llama_index/core/storage/docstore/types.py) for storing nodes.

 - `index_store`: A [`BaseIndexStore`](https://github.com/run-llama/llama_index/blob/main/llama-index-core/llama_index/core/storage/index_store/types.py#L13) for storing indices.

 - `vector_store`: A [`VectorStore`](https://github.com/run-llama/llama_index/blob/main/llama-index-core/llama_index/core/vector_stores/simple.py) for storing vectors.

 - `graph_store`: A [`GraphStore`](https://github.com/run-llama/llama_index/blob/main/llama-index-core/llama_index/core/graph_stores/simple.py) for storing knowledge graphs.

Below we instantiate the `StorageContext` from default settings indicating that we want to use a vector store.

In [12]:
from llama_index.core import StorageContext

# assign qdrant vector store to storage context
storage_context = StorageContext.from_defaults(
    vector_store=vector_store,
    )

In [13]:
# Import VectorStoreIndex for creating vector-based indexes
from llama_index.core import VectorStoreIndex

# Create the index with Qdrant storage
# from_documents() processes documents and stores embeddings in Qdrant
# Parameters:
# - document: List of Document objects to index
# - show_progress=True: Display progress bar during processing
# - store_nodes_override=True: Store nodes in docstore (for retrieval)
# - transformation=[sentence_splitter]: Custom node parser for chunking
# - embed_model: Embedding model to use for generating vectors
# - storage_context: StorageContext with Qdrant vector store
index = VectorStoreIndex.from_documents(
    document,  # List of documents to index
    show_progress=True,  # Show progress bar
    store_nodes_override=True,  # Store nodes for retrieval
    transformations=[sentence_splitter],  # Use custom node parser (note: plural "transformations")
    embed_model=embed_model,  # Embedding model
    storage_context=storage_context,  # Use Qdrant for storage
)

Parsing nodes:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/230 [00:00<?, ?it/s]

2026-01-28 18:00:03,696 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-01-28 18:00:03,697 - INFO - Retrying request to /embeddings in 0.486604 seconds
2026-01-28 18:00:04,947 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-01-28 18:00:04,950 - INFO - Retrying request to /embeddings in 0.762075 seconds
2026-01-28 18:00:06,050 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-01-28 18:00:06,053 - INFO - Retrying request to /embeddings in 1.741268 seconds
2026-01-28 18:00:08,105 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-01-28 18:00:08,105 - INFO - Retrying request to /embeddings in 3.566326 seconds
2026-01-28 18:00:11,971 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-01-28 18:00:11,971 - INFO - Retrying request 

KeyboardInterrupt: 

# 🪃 Retrieval

A `Retriever` is an interface exposed by the `Index`. An `Index` with its `Retriever` is used for storing and fetching data. The `Retriever` is a part of the `Index` and is used to retrieve the data stored in the Index.


### LlamaIndex provides [many different types of retrievers](https://github.com/run-llama/llama_index/tree/main/llama-index-core/llama_index/core/retrievers) to fetch relevant information from ingested data based on a given query. 

Some examples include

### Vector Retriever

The vector retriever uses vector similarity search to find the most relevant nodes (chunks of text) based on the query embedding. It requires a vector database like to store and search through the node embeddings.

### [Fusion Retriever](https://github.com/run-llama/llama_index/blob/main/llama-index-core/llama_index/core/retrievers/fusion_retriever.py)

The fusion retriever generates multiple queries from the original query, performs retrieval over an ensemble of retrievers for each query, and then fuses and reranks the results across all queries. This aims to better capture the query intent through query rewriting and ensembling.

### [Recursive Retriever](https://github.com/run-llama/llama_index/blob/main/llama-index-core/llama_index/core/retrievers/recursive_retriever.py)

The recursive retriever allows for hierarchical retrieval by first retrieving coarse nodes and then recursively retrieving finer-grained nodes within those coarse nodes. This can be useful for multi-level indexing and retrieval.

You can also combine retrievers in interesting ways and build out more advanced retrieval strategies, as we will see later in this course.


### In the example here, we're using a Vector Retriever

 - 🔍 When searching, your query is also converted into a vector embedding. 
 
- 🗂️ The `VectorStoreIndex` then performs a mathematical operation to rank embeddings based on semantic similarity to your query.

- 🔝 Top-k semantic retrieval is the simplest wasy to query a vector index.

- ⩬ You can also apply a similarity threshold  (e.g., only return results that are more similar than some value)


In [29]:
# Create a retriever from the index
# as_retriever() converts the index into a retriever for querying
# Parameters:
# - similarity_top_k: Return top K most similar results (default: 2)
# - similarity_threshold: Minimum similarity score (0-1), filters out low-quality results
# Note: Fixed typo "retirever" → "retriever"
retriever = index.as_retriever(
    similarity_top_k=5,  # Return top 5 most relevant chunks
    similarity_threshold=0.75  # Only return results with similarity >= 0.75
)

NameError: name 'index' is not defined

In [13]:
# Retrieve relevant documents/chunks for a query
# retrieve() converts the query to an embedding, searches Qdrant,
# and returns the most semantically similar nodes/chunks
# Returns a list of NodeWithScore objects (node + similarity score)
# Note: Fixed typo "retirever" → "retriever"
retriever.retrieve("What lessons can be learned from the poems about success?")

[NodeWithScore(node=TextNode(id_='1c76d824-ee82-4137-9d89-507d95c3c302', embedding=None, metadata={'file_path': 'data/pg10763.txt', 'file_name': 'pg10763.txt', 'file_type': 'text/plain', 'file_size': 405159, 'creation_date': '2024-05-21', 'last_modified_date': '2024-05-05'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='data/pg10763.txt', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'data/pg10763.txt', 'file_name': 'pg10763.txt', 'file_type': 'text/plain', 'file_size': 405159, 'creation_date': '2024-05-21', 'last_modified_date': '2024-05-05'}, hash='ce8443a693c1546f39c6bb0336c4d2f7929dd66c58288850e8b2df75ffa28edb'), <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='7510a8e9-9cba-4

But, chances are you don't just want the returned documents. You want the documents to be synthesized into a response. 

So, let's build on this pattern in the next lesson and see how we can get a response based on those retrieved documents.

In [30]:
# Close the Qdrant client connection
# This releases resources and closes the connection to Qdrant
# Important: Close connections when done to avoid resource leaks
# Note: The index data remains stored in Qdrant and can be accessed later
client.close()